# 02 — Fetch Products

Pulls the full OneBill product catalog and every product's price-plan
detail. Pure fetch/catalog step — matching against vBill plan codes happens
separately in `03_Match_Plan_Codes.ipynb`.

**Flow**

1. `GET /rest/ProductService/v1/products` — every product code.
2. `GET /rest/ProductService/v1/products/{code}` for each one.
   - If OneBill returns validation error `10PR1126` ("Product is not
     available for the user."), the product is **not** available to this
     API user — recorded (name + code) but with no usable price plans.
   - Otherwise, every entry in `pricePlanInfos[]` becomes one row of
     `df_available_priceplans`.

**Output**
- `migration_data/02_available_priceplans.csv`
- `migration_data/02_unavailable_products.csv`
- `migration_data/02_failed_product_lookups.csv`

## 1. Setup

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))

from onebill_common import *  # noqa: F401,F403

logger = get_logger("fetch_products")
session = new_session(max_workers=10)


python-dotenv could not parse statement starting at line 1
python-dotenv could not parse statement starting at line 5
python-dotenv could not parse statement starting at line 12
python-dotenv could not parse statement starting at line 17
python-dotenv could not parse statement starting at line 23
python-dotenv could not parse statement starting at line 29


## 2. Fetch the full product list

In [2]:
df_products_all = fetch_all_products(session)
logger.info(f"Fetched {len(df_products_all):,} products from ProductService")
df_products_all.head()


2026-07-21 12:38:30,805 [INFO] Fetched 20 products from ProductService


,name,categoryName,code,id,status
0,(AR) Voyager Voice Cloud Trunk,Cloud Voice - Demo AR,PROD709,812,ACTIVE
1,(AR) Voyager Voice Lite,Cloud Voice - Demo AR,PROD708,811,ACTIVE
2,(AR) Voyager Voice One,Cloud Voice - Demo AR,VOICE001,806,ACTIVE
3,(AR) Voyager Voice Premium,Cloud Voice - Demo AR,VOICE101,807,ACTIVE
4,AI Account,AI,PROD802,1002,ACTIVE


## 3. Fetch each product's detail

Separates products into available / unavailable / failed — see the module
docstring above for what each means.

> **Response shape**: the single-product endpoint returns the product
> object **directly** — `data` *is* the product, unlike the list endpoint
> which wraps everything in `{"product": [...]}`. `pricePlanInfos` sits
> right on `data`.

In [3]:
available_rows: list[dict] = []       # one row per (product, pricePlanInfo)
unavailable_rows: list[dict] = []      # one row per product with no usable plans
failed_rows: list[dict] = []

codes = df_products_all["code"].dropna().unique().tolist()
logger.info(f"Fetching detail for {len(codes):,} product codes...")

for i, code in enumerate(codes, start=1):
    status, data, message = fetch_product_detail(session, code)

    if status == "available":
        product = data or {}
        plans = product.get("pricePlanInfos", [])
        if not plans:
            unavailable_rows.append({
                "product_name": product.get("name"),
                "product_code": product.get("code", code),
                "reason": "available product, but no pricePlanInfos returned",
            })
        else:
            for plan in plans:
                available_rows.append({
                    "product_name":    product.get("name"),
                    "product_code":    product.get("code", code),
                    "priceplan_name":  plan.get("name"),
                    "priceplan_code":  plan.get("code"),
                    "priceplan_id":    plan.get("id"),
                })

    elif status == "unavailable":
        list_row = df_products_all.loc[df_products_all["code"] == code]
        name = list_row["name"].iloc[0] if not list_row.empty else None
        unavailable_rows.append({"product_name": name, "product_code": code, "reason": message})

    else:  # failed
        failed_rows.append({"product_code": code, "error": message})
        logger.error(f"  [FAIL] product {code} — {message}")

    if i % 50 == 0 or i == len(codes):
        logger.info(f"Progress: {i}/{len(codes)} product codes checked")

logger.info(
    f"Done — {len(available_rows):,} available price-plan rows, "
    f"{len(unavailable_rows):,} unavailable products, {len(failed_rows):,} failed lookups"
)


2026-07-21 12:38:30,840 [INFO] Fetching detail for 20 product codes...
2026-07-21 12:38:43,104 [INFO] Progress: 20/20 product codes checked
2026-07-21 12:38:43,106 [INFO] Done — 9 available price-plan rows, 17 unavailable products, 0 failed lookups


## 4. Build dataframes

In [4]:
df_available_priceplans = pd.DataFrame(
    available_rows, columns=["product_name", "product_code", "priceplan_name", "priceplan_code", "priceplan_id"]
)
df_unavailable_products = pd.DataFrame(unavailable_rows, columns=["product_name", "product_code", "reason"])
df_failed_products = pd.DataFrame(failed_rows, columns=["product_code", "error"])

logger.info(
    f"{len(df_available_priceplans):,} available price-plan rows, "
    f"{len(df_unavailable_products):,} unavailable products, "
    f"{len(df_failed_products):,} failed lookups"
)
df_available_priceplans.head(20)


2026-07-21 12:38:43,129 [INFO] 9 available price-plan rows, 17 unavailable products, 0 failed lookups


,product_name,product_code,priceplan_name,priceplan_code,priceplan_id
0,Wholesale Fibre BS2 (Chorus),PROD1302,WS Tail+Data - BS2 Res (Chorus) - 100/20,SC-02138,1703
1,Wholesale Fibre BS2 (Chorus),PROD1302,WS Tail+Data - BS2 Res (Chorus) - 500/100,SC-02140,1704
2,Wholesale Fibre BS2 (Chorus),PROD1302,WS Tail+Data - BS2 Res (Chorus) - 920/500,SC-02139,1705
3,Wholesale Fibre BS2 (Chorus),PROD1302,WS Tail+Data - BS2 Res Fibre Starter (Chorus) ...,SC-04046,1706
4,Wholesale Fibre BS2 (Enable),PROD1303,WS Tail+Data - BS2 Res (Enable) - 100/20,SC-02134,1707
5,Wholesale Fibre BS2 (Enable),PROD1303,WS Tail+Data - BS2 Res (Enable) - 500/100,SC-02133,1708
6,Wholesale Fibre BS2 (Enable),PROD1303,WS Tail+Data - BS2 Res (Enable) - 920/500,SC-02132,1709
7,Wholesale Fibre BS2 (Enable),PROD1303,WS Tail+Data - BS2 Res Fibre Starter (Enable) ...,SC-04045,1710
8,Wholesale Fibre BS2 (TFF),PROD1304,WS Tail+Data - BS2 Res Fibre Starter (TFF) - 1...,SC-04047,1711


## 5. Save

In [5]:
save_df("products_available", df_available_priceplans)
save_df("products_unavailable", df_unavailable_products)
save_df("products_failed", df_failed_products)


Saved 9 rows -> migration_data\02_available_priceplans.csv
Saved 17 rows -> migration_data\02_unavailable_products.csv
Saved 0 rows -> migration_data\02_failed_product_lookups.csv
